# Hiperparametre Optimizasyonu — U-Net + ResNet34

Ana çalışmadaki en iyi model (`unet_resnet34`, val Dice **0,8059**) üzerinde altı deney.
Her deney baseline'dan **yalnızca tek bir şeyi** değiştirir; hepsi aynı baseline'a karşı
karşılaştırılır. Kademeli (greedy) arama yapılmaz — her sonucun tek başına yorumlanabilmesi
ve seçim yanlılığının birikmemesi için.

| # | Deney | Değişen |
|---|---|---|
| 0 | `threshold` | Eğitim yok — mevcut `best.pt` ile eşik taraması |
| 1 | `no_vflip` | Dikey çevirme kapalı (p 0,3 → 0) |
| 2 | `enc_lr_3e5` | Encoder LR 5e-5 → 3e-5 |
| 3 | `enc_lr_1e5` | Encoder LR 5e-5 → 1e-5 |
| 4 | `cosine` | ReduceLROnPlateau → CosineAnnealingLR |
| 5 | `dice_focal` | Loss: Dice+BCE → Dice+Focal |
| 6 | `dice_tversky` | Loss: Dice+BCE → Dice+Tversky |

**Eşik taraması eğitim setinde yapılır**, validation'da değil — 89 vakalık validation'a
hiperparametre uydurmak sonucu şişirir. Eğitim seti 5 kat büyük ve sızıntı oluşturmaz.

**Devam edebilir**: biten deney atlanır. Oturum koparsa notebook baştan çalıştırılıp
kaldığı yerden sürdürülür.

Veri bölünmesi, augmentation (deney 1 hariç), loss (deney 5-6 hariç), seed ve değerlendirme
protokolü ana çalışmayla birebir aynıdır.

In [ ]:
!pip -q install segmentation-models-pytorch albumentations torchmetrics nibabel openpyxl

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, json, random, shutil, time, math
from pathlib import Path

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## CONFIG

In [ ]:
# ---- Baseline: ana calismadaki unet_resnet34 ayarlari ----
BASE = dict(
    arch='unet', encoder='resnet34',
    batch_size=16, lr_encoder=5e-5, lr_decoder=2.5e-4,
    max_epochs=70, patience=15,
    vflip_p=0.3, scheduler='plateau', loss='dice_bce',
)
BASELINE_DICE = 0.8059   # ana calismadan, karsilastirma icin

# ---- Deneyler: her biri baseline'dan TEK bir sey degistirir ----
EXPERIMENTS = {
    'no_vflip':     dict(vflip_p=0.0),
    'enc_lr_3e5':   dict(lr_encoder=3e-5),
    'enc_lr_1e5':   dict(lr_encoder=1e-5),
    # cosine cevrimini tamamlamali: erken durdurma ile kesilirse dusuk-LR
    # ince ayar fazina hic girmez ve haksiz yere kaybeder -> patience = max_epochs
    'cosine':       dict(scheduler='cosine', patience=70),
    'dice_focal':   dict(loss='dice_focal'),
    'dice_tversky': dict(loss='dice_tversky'),
}

SEED = 42
IMAGE_SIZE = 512
WARMUP_EPOCHS = 3
WEIGHT_DECAY = 1e-4

DATA_ROOT = '/content/drive/MyDrive/Mendeley_Data_Density_Entegrasyonu'
IMAGES_TR, LABELS_TR = f'{DATA_ROOT}/imagesTr', f'{DATA_ROOT}/labelsTr'
IMAGES_VAL, LABELS_VAL = f'{DATA_ROOT}/imagesVal', f'{DATA_ROOT}/labelsVal'

# Ana calismanin ciktilari (baseline best.pt buradan okunur)
MAIN_DIR = '/content/drive/MyDrive/density_segmentation/unet_resnet34'
BASE_CKPT = f'{MAIN_DIR}/checkpoints/best.pt'

# HPO ciktilari ayri klasorde -- ana calismanin dosyalarina dokunulmaz
HPO_ROOT = '/content/drive/MyDrive/density_segmentation_hpo'
Path(HPO_ROOT).mkdir(parents=True, exist_ok=True)
SUMMARY_XLSX = f'{HPO_ROOT}/hpo_summary.xlsx'

# Yeniden calistirilacak deneyler. Ornek: FORCE_RERUN = ['no_vflip', 'enc_lr_3e5']
# Listedekilerin onceki ciktisi silinir ve deney sifirdan kosar.
FORCE_RERUN = []

print('Baseline Dice (ana calisma):', BASELINE_DICE)
print('Deney sayisi:', len(EXPERIMENTS))
print('Cikti:', HPO_ROOT)

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

## Veri

In [ ]:
def build_case_index(images_dir, labels_dir):
    def stem(p):
        name = os.path.basename(p)
        if name.endswith('.nii.gz'):
            name = name[:-len('.nii.gz')]
        if name.endswith('_0000'):
            name = name[:-len('_0000')]
        return name
    img_map = {stem(p): p for p in sorted(glob.glob(f'{images_dir}/*.nii.gz'))}
    lbl_map = {stem(p): p for p in sorted(glob.glob(f'{labels_dir}/*.nii.gz'))}
    common = sorted(set(img_map) & set(lbl_map))
    return [(c, img_map[c], lbl_map[c]) for c in common]


train_index = build_case_index(IMAGES_TR, LABELS_TR)
val_index = build_case_index(IMAGES_VAL, LABELS_VAL)
print('Train:', len(train_index), '| Val:', len(val_index))
assert len(train_index) and len(val_index)

In [ ]:
# Ana calismayla birebir ayni normalizasyon (max_pixel_value=1.0 kritik)
def imagenet_normalize():
    return A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225), max_pixel_value=1.0)


def get_train_transform(size, vflip_p=0.3):
    return A.Compose([
        A.Resize(size, size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=vflip_p),          # <- deney 1'de 0.0
        A.Rotate(limit=15, p=0.5, border_mode=0),
        A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
        A.ElasticTransform(alpha=30, sigma=15, p=0.2),
        A.CoarseDropout(num_holes_range=(1, 4), hole_height_range=(8, 24), hole_width_range=(8, 24), p=0.2),
        imagenet_normalize(),
        ToTensorV2(),
    ])


def get_val_transform(size):
    return A.Compose([A.Resize(size, size), imagenet_normalize(), ToTensorV2()])


CACHE_DIR = '/content/npy_cache'
Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)


def load_volume(path, cache_key):
    cache_path = f'{CACHE_DIR}/{cache_key}.npy'
    if os.path.exists(cache_path):
        return np.load(cache_path)
    arr = nib.load(path).get_fdata().astype(np.float32)
    tmp = f'{cache_path}.{os.getpid()}.tmp'
    with open(tmp, 'wb') as f:
        np.save(f, arr)
    os.replace(tmp, cache_path)
    return arr


class DensityDataset(Dataset):
    def __init__(self, index, transform):
        self.index, self.transform = index, transform

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i):
        cid, ip, lp = self.index[i]
        img = load_volume(ip, f'{cid}_img')
        lbl = load_volume(lp, f'{cid}_lbl')
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        aug = self.transform(image=np.stack([img] * 3, -1), mask=lbl)
        m = aug['mask']
        if m.ndim == 2:
            m = m.unsqueeze(0)
        return aug['image'], m.float(), cid


def make_loaders(vflip_p, batch_size):
    tr = DataLoader(DensityDataset(train_index, get_train_transform(IMAGE_SIZE, vflip_p)),
                    batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
    va = DataLoader(DensityDataset(val_index, get_val_transform(IMAGE_SIZE)),
                    batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    return tr, va


# esik taramasi icin: egitim seti, augmentation'siz
train_eval_loader = DataLoader(DensityDataset(train_index, get_val_transform(IMAGE_SIZE)),
                               batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
val_eval_loader = DataLoader(DensityDataset(val_index, get_val_transform(IMAGE_SIZE)),
                             batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
print('Loader hazir.')

## Model, loss, metrik

In [ ]:
def build_model(cfg):
    kw = dict(encoder_name=cfg['encoder'], encoder_weights='imagenet', in_channels=3, classes=1)
    if cfg['arch'] == 'unetpp':
        return smp.UnetPlusPlus(**kw)
    return smp.Unet(**kw)


def build_optimizer(model, cfg):
    enc = list(model.encoder.parameters())
    enc_ids = {id(p) for p in enc}
    dec = [p for p in model.parameters() if id(p) not in enc_ids]
    return torch.optim.AdamW([
        {'params': enc, 'lr': cfg['lr_encoder']},
        {'params': dec, 'lr': cfg['lr_decoder']},
    ], weight_decay=WEIGHT_DECAY)


def build_scheduler(optimizer, cfg):
    if cfg['scheduler'] == 'cosine':
        # warmup sonrasi kalan epoch boyunca kosinus; her epoch step() cagrilir
        return torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=max(1, cfg['max_epochs'] - WARMUP_EPOCHS), eta_min=1e-7)
    return torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-7)

In [ ]:
# --- Loss varyantlari. Hepsi fp32'de hesaplanir (AMP altinda fp16 toplama tasar) ---

def _soft_dice(probs_flat, targets_flat, smooth=1.0):
    inter = (probs_flat * targets_flat).sum(1)
    return (2 * inter + smooth) / (probs_flat.sum(1) + targets_flat.sum(1) + smooth)


def dice_bce_loss(logits, targets, smooth=1.0):
    logits, targets = logits.float(), targets.float()
    bce = F.binary_cross_entropy_with_logits(logits, targets)
    p = torch.sigmoid(logits).reshape(logits.size(0), -1)
    t = targets.reshape(targets.size(0), -1)
    return bce + (1 - _soft_dice(p, t, smooth).mean())


def dice_focal_loss(logits, targets, alpha=0.25, gamma=2.0, smooth=1.0):
    logits, targets = logits.float(), targets.float()
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
    pt = torch.exp(-bce)                       # dogru sinif olasiligi
    a_t = alpha * targets + (1 - alpha) * (1 - targets)
    focal = (a_t * (1 - pt) ** gamma * bce).mean()
    p = torch.sigmoid(logits).reshape(logits.size(0), -1)
    t = targets.reshape(targets.size(0), -1)
    return focal + (1 - _soft_dice(p, t, smooth).mean())


def dice_tversky_loss(logits, targets, alpha=0.7, beta=0.3, smooth=1.0):
    # alpha > beta -> FP daha agir cezalandirilir.
    # Baseline modelde recall (0,8392) > precision (0,8153), yani hafif asiri
    # segmentasyon var; asimetriyi precision lehine kuruyoruz.
    logits, targets = logits.float(), targets.float()
    p = torch.sigmoid(logits).reshape(logits.size(0), -1)
    t = targets.reshape(targets.size(0), -1)
    tp = (p * t).sum(1)
    fp = (p * (1 - t)).sum(1)
    fn = ((1 - p) * t).sum(1)
    tversky = (tp + smooth) / (tp + alpha * fp + beta * fn + smooth)
    return (1 - tversky.mean()) + (1 - _soft_dice(p, t, smooth).mean())


LOSSES = {'dice_bce': dice_bce_loss, 'dice_focal': dice_focal_loss, 'dice_tversky': dice_tversky_loss}

In [ ]:
@torch.no_grad()
def compute_metrics(logits, targets, threshold=0.5, eps=1.0):
    preds = (torch.sigmoid(logits.float()) > threshold).float()
    targets = (targets > 0.5).float()
    p = preds.reshape(preds.size(0), -1)
    t = targets.reshape(targets.size(0), -1)
    tp = (p * t).sum(1); fp = (p * (1 - t)).sum(1); fn = ((1 - p) * t).sum(1)
    dice = ((2 * tp + eps) / (2 * tp + fp + fn + eps)).mean().item()
    return dict(dice=dice,
                iou=((tp + eps) / (tp + fp + fn + eps)).mean().item(),
                precision=((tp + eps) / (tp + fp + eps)).mean().item(),
                recall=((tp + eps) / (tp + fn + eps)).mean().item(),
                f1=dice)


@torch.no_grad()
def per_case_dice(model, loader, threshold=0.5, eps=1.0):
    model.eval()
    out = []
    for images, masks, _ in loader:
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        with torch.amp.autocast('cuda'):
            logits = model(images)
        p = (torch.sigmoid(logits.float()) > threshold).float().reshape(images.size(0), -1)
        t = (masks > 0.5).float().reshape(images.size(0), -1)
        tp = (p * t).sum(1); fp = (p * (1 - t)).sum(1); fn = ((1 - p) * t).sum(1)
        out += ((2 * tp + eps) / (2 * tp + fp + fn + eps)).cpu().tolist()
    return np.array(out)

## Deney 0 — Eşik taraması (eğitim gerektirmez)

Ana çalışmanın `best.pt`'si ile **eğitim seti** üzerinde eşik taranır, kazanan eşik
validation'a uygulanır. Validation'da taramak, sonucu aynı sette raporladığımız için
yapay kazanç üretirdi.

In [ ]:
thr_model = build_model(BASE).to(DEVICE)
_ck = torch.load(BASE_CKPT, map_location=DEVICE, weights_only=False)
_sd = {(k[6:] if k.startswith('model.') else k): v for k, v in _ck['model'].items()}
thr_model.load_state_dict(_sd)
thr_model.eval()
print('Baseline best.pt yuklendi (epoch', _ck['epoch'], ')')

THRESHOLDS = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]
rows = []
for th in THRESHOLDS:
    dtr = per_case_dice(thr_model, train_eval_loader, threshold=th)
    dva = per_case_dice(thr_model, val_eval_loader, threshold=th)
    rows.append(dict(threshold=th, train_dice=dtr.mean(), val_dice=dva.mean(),
                     val_sem=dva.std(ddof=1) / np.sqrt(len(dva))))
    print(f'  th={th:.2f}  train {dtr.mean():.4f}   val {dva.mean():.4f}')

thr_df = pd.DataFrame(rows)
best_th = float(thr_df.loc[thr_df.train_dice.idxmax(), 'threshold'])
val_at_best = float(thr_df.loc[thr_df.threshold == best_th, 'val_dice'].iloc[0])
val_at_050 = float(thr_df.loc[thr_df.threshold == 0.50, 'val_dice'].iloc[0])

print()
print(f'Egitim setinde en iyi esik : {best_th:.2f}')
print(f'  bu esikle val Dice       : {val_at_best:.4f}')
print(f'  esik 0.50 ile val Dice   : {val_at_050:.4f}')
print(f'  kazanc                   : {val_at_best - val_at_050:+.4f}')

thr_df.to_excel(f'{HPO_ROOT}/threshold_search.xlsx', index=False)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(thr_df.threshold, thr_df.train_dice, marker='o', label='Train (secim burada)')
ax.plot(thr_df.threshold, thr_df.val_dice, marker='s', label='Validation')
ax.axvline(best_th, ls='--', c='gray', lw=1)
ax.set_xlabel('Esik'); ax.set_ylabel('Dice'); ax.legend(); ax.grid(alpha=0.3)
ax.set_title('Esik taramasi — U-Net + ResNet34')
plt.tight_layout(); plt.savefig(f'{HPO_ROOT}/threshold_search.png', dpi=150); plt.show()

# --- baseline'in vaka-basi Dice'i ve metrikleri (esik 0.50) -> eslesmis test icin ---
base_pc = per_case_dice(thr_model, val_eval_loader, threshold=0.50)
np.save(f'{HPO_ROOT}/baseline_per_case_dice.npy', base_pc)

thr_model.eval()
_agg = dict(dice=0.0, iou=0.0, precision=0.0, recall=0.0, f1=0.0); _n = 0
with torch.no_grad():
    for images, masks, _ in val_eval_loader:
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        with torch.amp.autocast('cuda'):
            logits = thr_model(images)
        m = compute_metrics(logits, masks)
        for k in _agg:
            _agg[k] += m[k] * images.size(0)
        _n += images.size(0)
for k in _agg:
    _agg[k] /= _n

BASE_METRICS = dict(_agg,
                    dice_std=float(base_pc.std(ddof=1)),
                    dice_sem=float(base_pc.std(ddof=1) / np.sqrt(len(base_pc))),
                    best_epoch=int(_ck['epoch']))
print()
print('Baseline (bu notebookta yeniden olculdu):')
for k, v in BASE_METRICS.items():
    print(f'  {k:12} {v:.4f}' if isinstance(v, float) else f'  {k:12} {v}')
print(f'  ana calismada raporlanan Dice: {BASELINE_DICE:.4f}')

del thr_model; torch.cuda.empty_cache()

## Eğitim döngüsü

In [ ]:
SCHEMA = ['epoch', 'train_loss', 'val_loss', 'dice', 'iou', 'precision', 'recall', 'f1', 'lr']


def run_experiment(name, overrides):
    cfg = dict(BASE); cfg.update(overrides)
    out_dir = f'{HPO_ROOT}/{name}'
    ckpt_dir = f'{out_dir}/checkpoints'
    Path(ckpt_dir).mkdir(parents=True, exist_ok=True)
    log_path = f'{out_dir}/{name}_log.xlsx'
    best_path = f'{ckpt_dir}/best.pt'
    done_flag = f'{out_dir}/DONE.json'

    last_path = f'{ckpt_dir}/last.pt'

    if name in FORCE_RERUN:
        for _p in (done_flag, last_path, log_path):
            if os.path.exists(_p):
                os.remove(_p)
        print(f'[{name}] FORCE_RERUN -> onceki cikti silindi')

    if os.path.exists(done_flag):
        r = json.load(open(done_flag))
        print(f'[{name}] zaten tamamlanmis -> atlaniyor (dice {r["dice"]:.4f})')
        return r

    set_seed(SEED)
    train_loader, val_loader = make_loaders(cfg['vflip_p'], cfg['batch_size'])

    model = build_model(cfg).to(DEVICE)
    optimizer = build_optimizer(model, cfg)
    target_lrs = [g['lr'] for g in optimizer.param_groups]
    scheduler = build_scheduler(optimizer, cfg)
    scaler = torch.amp.GradScaler('cuda')
    loss_fn = LOSSES[cfg['loss']]

    # --- yarim kalmis kosu varsa kaldigi yerden devam ---
    start_epoch, best_dice, no_improve = 0, -1.0, 0
    if os.path.exists(last_path):
        _ck = torch.load(last_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(_ck['model'])
        optimizer.load_state_dict(_ck['optimizer'])
        scheduler.load_state_dict(_ck['scheduler'])
        scaler.load_state_dict(_ck['scaler'])
        torch.set_rng_state(_ck['torch_rng'].cpu())
        np.random.set_state(_ck['numpy_rng'])
        start_epoch = _ck['epoch'] + 1
        best_dice, no_improve = _ck['best_dice'], _ck['no_improve']
        if os.path.exists(log_path):   # loga cift satir girmesin
            _df = pd.read_excel(log_path)
            _df[_df['epoch'] < start_epoch][SCHEMA].to_excel(log_path, index=False)
        print(f'[{name}] YARIM KALMIS -> epoch {start_epoch} itibariyle devam '
              f'(best_dice {best_dice:.4f}, iyilesmeyen {no_improve})')
    else:
        if os.path.exists(log_path):
            os.remove(log_path)
        print(f'\n{"="*68}\n[{name}] basliyor | degisen: {overrides}\n{"="*68}')

    t0 = time.time()
    for epoch in range(start_epoch, cfg['max_epochs']):
        if epoch < WARMUP_EPOCHS:
            for g, base_lr in zip(optimizer.param_groups, target_lrs):
                g['lr'] = base_lr * (epoch + 1) / WARMUP_EPOCHS

        # --- train ---
        model.train(); tot, seen = 0.0, 0
        for images, masks, _ in train_loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda'):
                loss = loss_fn(model(images), masks)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer); scaler.update()
            tot += loss.item() * images.size(0); seen += images.size(0)
        train_loss = tot / seen

        # --- validate ---
        model.eval(); tot, seen = 0.0, 0
        agg = dict(dice=0.0, iou=0.0, precision=0.0, recall=0.0, f1=0.0)
        with torch.no_grad():
            for images, masks, _ in val_loader:
                images, masks = images.to(DEVICE), masks.to(DEVICE)
                with torch.amp.autocast('cuda'):
                    logits = model(images)
                    loss = loss_fn(logits, masks)
                tot += loss.item() * images.size(0); seen += images.size(0)
                m = compute_metrics(logits, masks)
                for k in agg:
                    agg[k] += m[k] * images.size(0)
        val_loss = tot / seen
        for k in agg:
            agg[k] /= seen

        if epoch >= WARMUP_EPOCHS:
            if cfg['scheduler'] == 'cosine':
                scheduler.step()
            else:
                scheduler.step(val_loss)
        lr_now = optimizer.param_groups[-1]['lr']

        row = dict(epoch=epoch, train_loss=train_loss, val_loss=val_loss, lr=lr_now, **agg)
        rows = pd.read_excel(log_path).to_dict('records') if os.path.exists(log_path) else []
        rows.append({k: row.get(k) for k in SCHEMA})
        pd.DataFrame(rows)[SCHEMA].to_excel(log_path, index=False)

        improved = agg['dice'] > best_dice
        if improved:
            best_dice, no_improve = agg['dice'], 0
            torch.save({'model': model.state_dict(), 'epoch': epoch, 'best_dice': best_dice,
                        'cfg': cfg}, best_path)
        else:
            no_improve += 1

        # kesintiye karsi tam durum (elektrik/oturum kopmasi)
        torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                    'scheduler': scheduler.state_dict(), 'scaler': scaler.state_dict(),
                    'epoch': epoch, 'best_dice': best_dice, 'no_improve': no_improve,
                    'torch_rng': torch.get_rng_state(), 'numpy_rng': np.random.get_state()},
                   last_path)

        print(f"  ep {epoch:03d} | train {train_loss:.4f} | val {val_loss:.4f} | "
              f"dice {agg['dice']:.4f} | best {best_dice:.4f}{' *' if improved else ''}")

        if no_improve >= cfg['patience']:
            print(f'  early stopping (patience {cfg["patience"]})')
            break

    # --- final: best.pt ile per-case Dice ---
    model.load_state_dict(torch.load(best_path, map_location=DEVICE, weights_only=False)['model'])
    d = per_case_dice(model, val_eval_loader)
    model.eval()
    agg = dict(dice=0.0, iou=0.0, precision=0.0, recall=0.0, f1=0.0); seen = 0
    with torch.no_grad():
        for images, masks, _ in val_eval_loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            with torch.amp.autocast('cuda'):
                logits = model(images)
            m = compute_metrics(logits, masks)
            for k in agg:
                agg[k] += m[k] * images.size(0)
            seen += images.size(0)
    for k in agg:
        agg[k] /= seen

    res = dict(experiment=name, changed=json.dumps(overrides), **agg,
               dice_std=float(d.std(ddof=1)), dice_sem=float(d.std(ddof=1) / np.sqrt(len(d))),
               best_epoch=int(torch.load(best_path, map_location='cpu', weights_only=False)['epoch']),
               minutes=round((time.time() - t0) / 60, 1))
    json.dump(res, open(done_flag, 'w'), indent=1)
    np.save(f'{out_dir}/per_case_dice.npy', d)
    print(f"[{name}] BITTI  dice {res['dice']:.4f}  (baseline {BASELINE_DICE:.4f}, "
          f"fark {res['dice']-BASELINE_DICE:+.4f})  {res['minutes']} dk")
    del model; torch.cuda.empty_cache()
    return res

## Mevcut sonuçların bütünlük kontrolü

Elektrik/oturum kesintisi yaşandıysa hangi deneyin sağlam bittiğini gösterir.
Eğitim yapmaz, yalnızca Drive'daki logları okur.

In [15]:
def check_integrity():
    rows = []
    for name in EXPERIMENTS:
        d = f'{HPO_ROOT}/{name}'
        done, log = f'{d}/DONE.json', f'{d}/{name}_log.xlsx'
        r = dict(deney=name, DONE=os.path.exists(done))
        if not os.path.exists(log):
            r['durum'] = 'log yok'
            rows.append(r)
            continue
        df = pd.read_excel(log)
        eps = df['epoch'].tolist()
        r['satir'] = len(eps)
        r['epoch'] = f'{min(eps)}-{max(eps)}' if eps else '-'
        eksik = sorted(set(range(min(eps), max(eps) + 1)) - set(eps)) if eps else []
        tekrar = len(eps) - len(set(eps))
        r['eksik'], r['tekrar'] = len(eksik), tekrar
        sorun = []
        if eksik:
            sorun.append(f'{len(eksik)} epoch eksik')
        if tekrar:
            sorun.append(f'{tekrar} tekrar satir')
        if r['DONE']:
            res = json.load(open(done))
            be = res['best_epoch']
            r['best_epoch'] = be
            if be in eps:
                if abs(float(df.loc[df['epoch'] == be, 'dice'].iloc[0]) - res['dice']) > 5e-3:
                    sorun.append('DONE.json dice logla uyusmuyor')
            else:
                sorun.append('best_epoch logda yok')
        else:
            sorun.append('tamamlanmamis')
        r['durum'] = 'SAGLAM' if not sorun else ' | '.join(sorun)
        rows.append(r)
    return pd.DataFrame(rows)


chk = check_integrity()
print(chk.to_string(index=False))

bozuk = chk.loc[chk.durum != 'SAGLAM', 'deney'].tolist()
print()
if bozuk:
    print('Sorunlu deneyler:', bozuk)
    print('Yeniden calistirmak icin CONFIG hucresinde su satiri guncelle:')
    print(f'  FORCE_RERUN = {bozuk}')
    print('sonra CONFIG hucresini ve deney hucresini tekrar calistir.')
else:
    print('Tum deneyler saglam: epoch dizileri kesintisiz, DONE.json loglarla tutarli.')
    print('Yeniden egitime gerek yok.')


       deney  DONE  satir epoch  eksik  tekrar  best_epoch  durum
    no_vflip  True     70  0-69      0       0          59 SAGLAM
  enc_lr_3e5  True     64  0-63      0       0          48 SAGLAM
  enc_lr_1e5  True     70  0-69      0       0          63 SAGLAM
      cosine  True     70  0-69      0       0          50 SAGLAM
  dice_focal  True     62  0-61      0       0          46 SAGLAM
dice_tversky  True     44  0-43      0       0          28 SAGLAM

Tum deneyler saglam: epoch dizileri kesintisiz, DONE.json loglarla tutarli.
Yeniden egitime gerek yok.


## Altı deneyi sırayla çalıştır

Oturum koparsa bu hücreyi tekrar çalıştırmak yeterli — biten deneyler atlanır.

In [ ]:
results = []
for name, ov in EXPERIMENTS.items():
    results.append(run_experiment(name, ov))

print('\nTum deneyler tamamlandi.')

## Karşılaştırma

In [ ]:
from scipy.stats import wilcoxon

base_pc = np.load(f'{HPO_ROOT}/baseline_per_case_dice.npy')

rows = [dict(experiment='BASELINE', changed='-', **BASE_METRICS, minutes=None,
             fark=0.0, paired_sem=float('nan'), wilcoxon_p=float('nan'), sonuc='referans')]

for r in results:
    pc = np.load(HPO_ROOT + '/' + r['experiment'] + '/per_case_dice.npy')
    diff = pc - base_pc                      # pozitif = deney baseline'dan iyi
    psem = float(diff.std(ddof=1) / np.sqrt(len(diff)))
    p = float(wilcoxon(pc, base_pc).pvalue) if bool((diff != 0).any()) else 1.0
    if p < 0.05 and diff.mean() > 0:
        verdict = 'ANLAMLI ARTIS'
    elif p < 0.05:
        verdict = 'ANLAMLI DUSUS'
    else:
        verdict = 'fark yok'
    rows.append(dict(r, fark=r['dice'] - BASE_METRICS['dice'],
                     paired_sem=psem, wilcoxon_p=p, sonuc=verdict))

df = pd.DataFrame(rows)
df['R_minus_P'] = df['recall'] - df['precision']
cols = ['experiment', 'changed', 'dice', 'fark', 'paired_sem', 'wilcoxon_p', 'sonuc',
        'iou', 'precision', 'recall', 'R_minus_P', 'dice_sem', 'best_epoch', 'minutes']
df = df[cols].sort_values('dice', ascending=False).reset_index(drop=True)
df.to_excel(SUMMARY_XLSX, index=False)

pd.set_option('display.width', 220)
print(df.to_string(index=False))
print()
print('Kaydedildi:', SUMMARY_XLSX)
print()
print('YORUM NOTU')
print('-' * 70)
print('Eslesmis karsilastirma ayni 89 vakada, vaka basina fark uzerinden yapildi;')
print('vakalar arasi zorluk farki sadelestigi icin kucuk gercek farklar gorulebiliyor.')
print()
print('Dikkat: alti deney denenip en iyisi secildiginde beklenen sahte kazanc')
print('~+0.018 Dice (SEM 0.0143, bagimsizlik varsayimiyla ust sinir). Wilcoxon p')
print('degeri TEK karsilastirma icin gecerlidir; alti karsilastirma birden')
print('yapildiginda coklu-karsilastirma duzeltmesi gerekir:')
print('  Bonferroni esigi = 0.05 / 6 = 0.0083')
print()
print('Bir kazancin gercek oldugunu dogrulamanin en saglam yolu, kazanan')
print('konfigurasyonu U-Net++ + ResNet34 uzerinde tekrarlamaktir:')
print("  EXPERIMENTS['dogrulama'] = dict(arch='unetpp', batch_size=12, <kazanan ayar>)")
print('Ayni yonde kazanc verirse gercek, vermezse validation gurultusu.')


In [ ]:
# Deneylerin Dice karsilastirmasi
fig, ax = plt.subplots(figsize=(9, 4.5))
d = df.sort_values('dice')
colors = ['#2c5aa0' if e == 'BASELINE' else '#5b8c5a' for e in d.experiment]
ax.barh(d.experiment, d.dice, color=colors)
ax.errorbar(d.dice, range(len(d)), xerr=d.dice_sem, fmt='none', ecolor='#333', capsize=3, lw=1)
ax.axvline(BASE_METRICS['dice'], ls='--', c='#2c5aa0', lw=1.2, label='Baseline')
ax.set_xlim(0.70, max(0.83, d.dice.max() + 0.02))
ax.set_xlabel('Val Dice (hata cubugu = SEM)'); ax.legend(); ax.grid(axis='x', alpha=0.3)
ax.set_title('Hiperparametre deneyleri — U-Net + ResNet34')
plt.tight_layout(); plt.savefig(f'{HPO_ROOT}/hpo_comparison.png', dpi=150); plt.show()